In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [2]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [3]:
from rag_helper import RAGBase


instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [4]:
answer = assistant.rag('How do I run Ollama locally?')
print(answer)

To run Ollama locally:

1. Install Ollama from https://ollama.com/download  
2. Open a terminal and run:

```bash
ollama run llama3
```

This will download the LLaMA 3 model, start it locally, and open a chat-like interface.

To check that the local server is running, you can also use:

```bash
curl http://localhost:11434
```

If needed, you can install the Python client with:

```bash
pip install ollama
```


In [5]:
answer = assistant.rag('How do I run Olama locally?')
print(answer)

I don’t have a FAQ entry for running **Ollama/Olama locally** in the provided context.


In [6]:
messages = [
    {'role': 'user', 'content': 'I just discovered the course. Can I join it?'}
]

response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
)

response.output_text

'Absolutely — in most cases, yes, you can join even if you’ve just discovered it.\n\nA few things to check:\n- **Whether enrollment is still open**\n- **Any prerequisites** or required background\n- **Whether there’s a registration deadline**\n- **If the course has a waiting list**\n\nIf you want, I can help you figure out the next step. Just send me:\n- the **course name or link**, and\n- where you’re seeing it offered (school, platform, event, etc.)\n\nThen I can help you determine if you can still join and what to do next.'

In [7]:
def search(query):
    boost_dict = {'question': 3.0, 'section': 0.5}
    filter_dict = {'course': 'llm-zoomcamp'}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [8]:
search_tool = {
    "type": "function",
    'name': 'search',
    'description': 'Search the FAQ database for entries matching the given query.',
    'parameters': {
        "type": "object",
        "properties": {
            'query': {
                "type": "string",
                'description': 'Search query text to look up in the course FAQ.'
            }
        },
        "required": ["query"],
        'additionalProperties': False
    }
}

In [9]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [10]:
len(response.output)

1

In [11]:
call = response.output[0]

In [12]:
call

ResponseFunctionToolCall(arguments='{"query":"Can I join the course after it has started? discovered the course join late enrollment"}', call_id='call_0ERNNzrEYd26MnqvlVbW0S9c', name='search', type='function_call', id='fc_07cad814bf73ab68006a276e05c194819fa63ac6941587e99c', namespace=None, status='completed')

In [13]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [16]:
len(response.output)

1

In [17]:
call = response.output[0]

In [18]:
call

ResponseFunctionToolCall(arguments='{"query":"join the course discovered late can I still enroll late registration join after course started"}', call_id='call_IE9oz62zVqSENlLIdyRyqD5e', name='search', type='function_call', id='fc_012540ff1b042fc5006a276fb759f48192b9307347f7353b04', namespace=None, status='completed')

In [20]:
import json
args = json.loads(call.arguments)

In [23]:
results = search(**args)

In [26]:
result_json = json.dumps(results,indent=2)
print(result_json)

[
  {
    "id": "74eb249bbf",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "I just discovered the course. Can I still join?",
    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions."
  },
  {
    "id": "69d122f12e",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "Certificate: Can I follow the course in a self-paced mode and get a certificate?",
    "answer": "No, you can only get a certificate if you finish the course with a \"live\" cohort.\n\nWe don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled."
  },
  {
    "id": "9f689c185f",
    "course": "llm-zoomcamp",
    "section": "General Course

In [27]:
function_call_output={
    'type': 'function_call_output',
    'call_id': call.call_id,
    'output': result_json
}

In [33]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'}]

In [34]:
messages.append(call)

In [35]:
messages.append(function_call_output)

In [37]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [38]:
print(response.output_text)

Yes — you can still join the course.

If you want a certificate, you’ll need to submit your project while submissions are still open.


In [40]:
usage = response.usage
usage.input_tokens, usage.output_tokens, usage.total_tokens

(620, 32, 652)

In [ ]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    # Prices per 1M tokens (example pricing)
    INPUT_PRICE_PER_MILLION = 0.15   # $0.15 / 1M input tokens
    OUTPUT_PRICE_PER_MILLION = 0.60  # $0.60 / 1M output tokens

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION

    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost
    }


# Your tokens
result = calculate_gpt54mini_price(652, 33)

print("Total Cost: $", round(result["total_cost"], 8))